# Stage A — Building the corpus
## Database exports → `data/corpus.csv`

**AI adoption in urban planning governance: a systematic review**
Lartey & Law (2025), *Landscape and Urban Planning* 258, 105337
DOI: [10.1016/j.landurbplan.2025.105337](https://doi.org/10.1016/j.landurbplan.2025.105337)

---

This notebook turns raw database exports into the single `corpus.csv` that notebooks
02–04 read. It covers Section 3.1 of the paper: the search across PubMed,
ScienceDirect and Consensus, the merge into one dataset, and the keyword-based
selection that took the pool from **3,715 retrieved** to **588 included** records.

### Bring your own exports

The corpus behind the published paper is our own collected dataset and is not
distributed here. Drop your `.ris` or `.csv` exports into `data/raw_ris/` and
`data/raw_csv/` and run the stages in order — you get a `corpus.csv` with the
structure the rest of the pipeline expects. All paths are relative, so nothing needs
editing first.

If you have no exports to hand, Stage A1 generates a small synthetic set so you can
watch the pipeline run before committing your own search to it.

### The output contract

| Column | Required | Notes |
|---|---|---|
| `Title` | yes | |
| `Abstract` | yes | may be blank on some rows |
| `Keyword` | yes | semicolon separated |
| `Year` | yes | integer |
| `Journal` | yes | used by notebook 02 |
| `Authors` | no | semicolon separated |
| `Affiliation` | no | strongly recommended — notebook 03 uses it |
| `DOI` | no | bare identifier |
| `StudyType` | no | if you code it during screening |
| `SourceFile` | no | provenance |

`Affiliation` is worth chasing down in your export settings. Without it notebook 03
can only tell you what the literature is *about*, not where its authors are based —
and for a review whose findings concern geographic representation, those are two
different claims.

---
## A0 — Setup

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


def _repo_relative(p):
    """Resolve whether the kernel started in notebooks/ or the repository root."""
    p = Path(p)
    if p.exists():
        return p
    alt = Path(str(p).replace("../", "", 1))
    return alt if (alt.exists() or Path("notebooks").is_dir()) else p


CONFIG = {
    "data_mode":   "demo",      # "demo" | "real"
    "data_dir":    "../data",
    "output_dir":  "../outputs",
    "seed":        42,
}

SEED = CONFIG["seed"]
DATA_DIR = _repo_relative(CONFIG["data_dir"])
OUT_DIR = _repo_relative(CONFIG["output_dir"])
RIS_DIR = DATA_DIR / "raw_ris"
CSV_DIR = DATA_DIR / "raw_csv"
INTERIM = DATA_DIR / "interim"
for d in (DATA_DIR, RIS_DIR, CSV_DIR, INTERIM, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# The four extraction keywords from Section 3.1.3. These are what took the pool
# from 3,715 retrieved to 588 included. Edit freely - nothing downstream hard-codes them.
EXTRACTION_KEYWORDS = {
    "Urban planning":          ["urban planning", "city planning"],
    "Decision-making":         ["decision-making", "decision making"],
    "Policy making":           ["policy making", "policymaking", "policy formulation"],
    "Artificial Intelligence": ["artificial intelligence", " ai ", "machine learning",
                                "deep learning"],
}

# Application areas and disciplines (Sections 3.2.1, 4.4), with the paper's counts
# as comparison targets. Notebook 02 classifies against these.
APPLICATION_AREAS = {
    "Community and Social Services":          ["community", "social service", "public service",
                                               "citizen", "wellbeing", "well-being", "equity"],
    "Environmental Planning and Sustainability": ["environment", "sustainab", "climate",
                                                  "green", "ecolog", "emission"],
    "Smart Cities and Infrastructure Development": ["smart city", "smart cities",
                                                    "infrastructure", "iot", "digital twin"],
    "Land Use and Zoning":                    ["land use", "land-use", "zoning",
                                               "urban form", "spatial planning"],
    "Transportation Planning and Management": ["transport", "mobility", "traffic",
                                               "autonomous vehicle", "logistics"],
}
PAPER_APPLICATION_COUNTS = {"Community and Social Services": 278,
                            "Environmental Planning and Sustainability": 148,
                            "Smart Cities and Infrastructure Development": 69,
                            "Land Use and Zoning": 58,
                            "Transportation Planning and Management": 35}

DISCIPLINES = {
    "Public Administration": ["public administration", "governance", "policy",
                              "government", "public sector"],
    "Social Sciences":       ["social", "society", "sociolog", "behaviour", "behavior",
                              "community"],
    "Urban Studies/Planning": ["urban studies", "urban planning", "planning", "urbanism"],
    "Computer Science":      ["computer science", "algorithm", "computing", "software",
                              "data science"],
    "Engineering":           ["engineering", "systems", "control", "optimi"],
    "Law & Regulation":      ["law", "legal", "regulation", "compliance"],
    "Economics":             ["econom", "market", "cost-benefit"],
}
PAPER_DISCIPLINE_COUNTS = {"Public Administration": 249, "Social Sciences": 234,
                           "Other": 40, "Engineering": 31, "Computer Science": 23,
                           "Urban Studies/Planning": 7, "Law & Regulation": 2,
                           "Economics": 1}

# Published funnel (Section 3.1.3) - compare your run against these
PAPER_FUNNEL = {"retrieved": 3715, "included": 588, "countries": 83}

print(f"data   -> {DATA_DIR.resolve()}")
print(f"outputs-> {OUT_DIR.resolve()}")
print(f"\nExtraction keywords: {list(EXTRACTION_KEYWORDS)}")
print(f"Paper funnel: {PAPER_FUNNEL['retrieved']:,} retrieved -> "
      f"{PAPER_FUNNEL['included']} included, {PAPER_FUNNEL['countries']} countries")
ris_files = sorted(RIS_DIR.glob("*.ris"))
csv_files = sorted(CSV_DIR.glob("*.csv"))
print(f"\nRIS files: {len(ris_files)}   CSV files: {len(csv_files)}")
for f in ris_files + csv_files:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

---
## A1 — Read the exports

RIS is line-oriented: `TAG  - value`, with continuation lines for wrapped fields and
`ER  -` closing each record. Reading it by flattening newlines and regex-matching
looks simpler but truncates any abstract containing a capital-letter pair followed by
two spaces and a dash, which does happen. The parser below reads line by line, keeps
continuations attached to their field, and counts anything it cannot interpret so
nothing disappears quietly.

Scopus and Web of Science CSV exports are also read, and their column names
normalised to the same schema.

In [ ]:
RIS_FIELDS = {"TY": "Type", "T1": "Title", "TI": "Title", "AU": "Authors",
              "JO": "Journal", "JF": "Journal", "T2": "Journal", "PY": "Year",
              "DA": "Date", "AB": "Abstract", "N2": "Abstract", "KW": "Keyword",
              "DO": "DOI", "UR": "URL", "AD": "Affiliation", "C1": "Affiliation",
              "SN": "ISSN", "VL": "Volume", "IS": "Issue", "PB": "Publisher"}
MULTI = {"AU", "KW", "AD", "C1"}
_LINE = re.compile(r"^([A-Z][A-Z0-9])\s{2}-\s?(.*)$")


def parse_ris(path):
    """Line-oriented RIS parser. Returns (records, stats)."""
    records, record, last, stats = [], {}, None, Counter()
    with open(path, "r", encoding="utf-8-sig", errors="replace") as fh:
        for raw in fh:
            line = raw.rstrip("\r\n")
            if not line.strip():
                continue
            m = _LINE.match(line)
            if m:
                tag, val = m.group(1), m.group(2).strip()
                if tag == "ER":
                    if record:
                        records.append(record); stats["records"] += 1
                    record, last = {}, None
                    continue
                if tag == "TY" and record:          # record opened without closing ER
                    records.append(record); stats["records"] += 1
                    stats["missing_ER"] += 1; record = {}
                name = RIS_FIELDS.get(tag)
                if name is None:
                    stats[f"unmapped:{tag}"] += 1; last = None; continue
                if tag in MULTI:
                    record[name] = f"{record[name]}; {val}" if name in record else val
                else:
                    record.setdefault(name, val)
                last = name
            elif last:
                record[last] = f"{record[last]} {line.strip()}".strip()
                stats["continuation_lines"] += 1
            else:
                stats["orphan_lines"] += 1
    if record:
        records.append(record); stats["records"] += 1; stats["missing_ER"] += 1
    return records, stats


# Scopus / WoS / ScienceDirect CSV headers -> our schema
CSV_ALIASES = {
    "title": "Title", "document title": "Title", "article title": "Title",
    "abstract": "Abstract",
    "author keywords": "Keyword", "keywords": "Keyword", "index keywords": "Keyword",
    "year": "Year", "publication year": "Year",
    "source title": "Journal", "journal": "Journal", "publication title": "Journal",
    "authors": "Authors", "author full names": "Authors", "author names": "Authors",
    "affiliations": "Affiliation", "authors with affiliations": "Affiliation",
    "addresses": "Affiliation",
    "doi": "DOI", "di": "DOI",
}


def read_export_csv(path):
    df = pd.read_csv(path, dtype=str, encoding_errors="replace",
                     on_bad_lines="skip", low_memory=False)
    ren = {c: CSV_ALIASES[c.strip().lower()] for c in df.columns
           if c.strip().lower() in CSV_ALIASES}
    df = df.rename(columns=ren)
    # A file may map two source columns onto one target; keep the first non-empty.
    df = df.loc[:, ~df.columns.duplicated()]
    df["SourceFile"] = path.name
    return df

In [ ]:
def synthesize_exports(n=900, seed=SEED):
    """Small synthetic corpus so the pipeline is runnable without real exports.

    Not the study's data. Shapes only: a post-2020 growth curve, a long-tailed
    journal distribution, and abstracts that mention places so notebook 03 has
    something to geocode.
    """
    rng = np.random.default_rng(seed)
    journals = ["Cities", "Landscape and Urban Planning", "Sustainable Cities and Society",
                "Technology in Society", "Journal of Urban Technology", "Habitat International",
                "Computers Environment and Urban Systems", "Urban Studies",
                "Government Information Quarterly", "Land Use Policy"]
    jp = np.array([12, 10, 9, 7, 6, 5, 5, 4, 4, 3], dtype=float); jp /= jp.sum()
    places = ["Amsterdam, Netherlands", "Singapore", "Shenzhen, China", "Barcelona, Spain",
              "Accra, Ghana", "Nairobi, Kenya", "Sao Paulo, Brazil", "Toronto, Canada",
              "Melbourne, Australia", "Seoul, South Korea", "Lagos, Nigeria",
              "Costa Rica", "Bogota, Colombia", "Mumbai, India", "Dubai"]
    types = ["Empirical", "Conceptual", "Review", "Case study", "Modelling"]
    years = np.arange(2005, 2025)
    w = np.exp((years - 2005) / 4.2); w /= w.sum()          # post-2020 surge

    rows = []
    for i in range(n):
        theme = rng.choice(list(EXTRACTION_KEYWORDS))
        terms = list(rng.choice(EXTRACTION_KEYWORDS[theme], size=2, replace=False))
        place = rng.choice(places)
        rows.append({
            "Title": f"{terms[0].capitalize()} for {terms[1]} in urban governance ({i:04d})",
            "Abstract": (f"This paper examines {terms[0]} and its role in {terms[1]} "
                         f"within urban planning. Evidence is drawn from {place}. "
                         f"We consider implications for equity, transparency and "
                         f"institutional capacity in smart city programmes."),
            "Keyword": "; ".join(terms + ["urban planning", "governance"]),
            "Year": str(rng.choice(years, p=w)),
            "Journal": str(rng.choice(journals, p=jp)),
            "Authors": f"Author {rng.integers(1, 220)}; Author {rng.integers(1, 220)}",
            "Affiliation": f"Department of Urban Planning, University of {place}",
            "DOI": f"10.9999/demo.{i:05d}",
            "StudyType": str(rng.choice(types)),
            "SourceFile": "SYNTHETIC",
        })
    return pd.DataFrame(rows)


frames, parse_stats = [], {}

for f in ris_files:
    recs, st = parse_ris(f)
    if recs:
        d = pd.DataFrame(recs); d["SourceFile"] = f.name
        frames.append(d); parse_stats[f.name] = dict(st)
        print(f"  {f.name}: {st['records']:>5} records, "
              f"{st['continuation_lines']} continuations, {st['orphan_lines']} orphans")

for f in csv_files:
    try:
        d = read_export_csv(f)
        frames.append(d)
        print(f"  {f.name}: {len(d):>5} rows, mapped {sorted(set(d.columns) & set(CSV_ALIASES.values()))}")
    except Exception as exc:
        print(f"  {f.name}: FAILED - {exc}")

if frames:
    raw = pd.concat(frames, ignore_index=True)
    print(f"\nTotal read: {len(raw):,} records from {len(frames)} file(s)")
elif CONFIG["data_mode"] == "demo":
    print("!" * 70)
    print("DEMO MODE - no exports found, generating a synthetic corpus.")
    print("This is NOT the study's data. Replace it with your own exports.")
    print("!" * 70)
    raw = synthesize_exports()
else:
    raise FileNotFoundError(
        f"No .ris files in {RIS_DIR} and no .csv files in {CSV_DIR}.\n"
        f"Add your exports, or set CONFIG['data_mode'] = 'demo'.")

raw.to_csv(INTERIM / "01_parsed.csv", index=False)
display(raw.head(3))

---
## A2 — Normalise

Map every source onto one schema. `Year` comes from the year field and falls back to
the first four-digit number in a date string; DOIs are stripped to the bare identifier
so deduplication does not treat `10.1016/x` and `https://doi.org/10.1016/x` as two
different papers.

In [ ]:
def normalise(df):
    out = pd.DataFrame(index=df.index)
    for col in ["Title", "Abstract", "Keyword", "Authors", "Journal",
                "Affiliation", "StudyType", "SourceFile"]:
        out[col] = (df[col] if col in df.columns else "")
        out[col] = out[col].fillna("").astype(str).str.strip()

    year = pd.to_numeric(df.get("Year"), errors="coerce")
    if "Date" in df.columns:
        alt = pd.to_numeric(df["Date"].astype(str).str.extract(r"(\d{4})")[0],
                            errors="coerce")
        year = year.fillna(alt)
    out["Year"] = year

    doi = df.get("DOI", pd.Series("", index=df.index))
    out["DOI"] = (doi.fillna("").astype(str).str.strip().str.lower()
                  .str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True)
                  .str.replace(r"^doi:\s*", "", regex=True).str.strip())
    return out


records = normalise(raw)
print(f"Records: {len(records):,}")
for col, label in [("DOI", "with a DOI"), ("Abstract", "with an abstract"),
                   ("Affiliation", "with an affiliation"), ("Journal", "with a journal")]:
    n = (records[col] != "").sum()
    print(f"  {label:22s} {n:>6,}  ({n / len(records):.0%})")
print(f"  {'with a year':22s} {records['Year'].notna().sum():>6,}")
if records["Year"].notna().any():
    print(f"  {'year range':22s} {int(records['Year'].min())}-{int(records['Year'].max())}")

if (records["Affiliation"] == "").mean() > 0.5:
    print("\n  Note: most records have no affiliation. Notebook 03 will be able to")
    print("  report what studies are ABOUT but not where their authors are based.")
    print("  Re-export with affiliations included if that distinction matters to you.")

records.to_csv(INTERIM / "02_normalised.csv", index=False)

---
## A3 — Deduplicate

Two passes. DOI is exact and goes first; records without one fall through to a
normalised-title match (lowercased, punctuation stripped, whitespace collapsed).
Every removal is counted so the funnel is checkable rather than reconstructed.

In [ ]:
def norm_title(s):
    return (s.astype(str).str.lower()
             .str.replace(r"[^a-z0-9 ]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())


ledger = [{"step": "Records retrieved", "removed": 0, "remaining": len(records)}]
d = records.copy()

has_doi = d["DOI"] != ""
drop = d[has_doi].index[d[has_doi].duplicated(subset=["DOI"], keep="first")]
d = d.drop(index=drop)
ledger.append({"step": "Duplicates removed (DOI)", "removed": len(drop), "remaining": len(d)})

d["_t"] = norm_title(d["Title"])
dup = d["_t"].ne("") & d.duplicated(subset=["_t"], keep="first")
n_t = int(dup.sum())
d = d[~dup].drop(columns="_t")
ledger.append({"step": "Duplicates removed (title)", "removed": n_t, "remaining": len(d)})

deduped = d.reset_index(drop=True)
print(f"{len(records):,} -> {len(deduped):,} after removing "
      f"{len(drop):,} DOI and {n_t:,} title duplicates")
deduped.to_csv(INTERIM / "03_deduplicated.csv", index=False)
display(pd.DataFrame(ledger))

---
## A4 — Screen

Section 3.1.3 describes parsing the retrieved pool with four extraction keywords —
*urban planning*, *decision-making*, *policy making*, *artificial intelligence* — which
narrowed 3,715 records to 588. That step is implemented here, preceded by basic
validity rules.

Inclusion requires engagement with **both** AI and the urban/governance domain. The
intersection is what defines the review, so a paper on machine learning in medicine
and a paper on zoning law both fall out here.

Full-text screening is a human step and is not automated; the ledger leaves a row for
you to fill in. Records are also tagged with every extraction keyword they match,
which notebook 04 uses for the theme matrix.

In [ ]:
AI_TERMS = ["artificial intelligence", "machine learning", "deep learning",
            "neural network", "algorithm", "autonomous", "predictive analytics",
            "automation", "computer vision", "natural language processing",
            "reinforcement learning", "generative ai", "large language model"]
URBAN_TERMS = ["urban", "city", "cities", "municipal", "metropolitan", "planning",
               "governance", "public administration", "policy", "civic",
               "spatial", "land use", "smart city"]


def contains_any(series, terms):
    return series.str.lower().str.contains(
        "|".join(re.escape(t) for t in terms), regex=True, na=False)


def blob_of(df):
    return (df["Title"] + " " + df["Abstract"] + " " + df["Keyword"])


s = deduped.copy()

n = len(s); s = s[s["Title"].str.len() > 10]
ledger.append({"step": "Excluded: no usable title", "removed": n - len(s), "remaining": len(s)})

n = len(s); s = s[s["Year"].notna() & s["Year"].between(1900, 2100)]
ledger.append({"step": "Excluded: no usable year", "removed": n - len(s), "remaining": len(s)})

n = len(s); s = s[contains_any(blob_of(s), AI_TERMS)]
ledger.append({"step": "Excluded: no AI content", "removed": n - len(s), "remaining": len(s)})

n = len(s); s = s[contains_any(blob_of(s), URBAN_TERMS)]
ledger.append({"step": "Excluded: no urban/governance content",
               "removed": n - len(s), "remaining": len(s)})

ledger.append({"step": "Excluded: full-text screening (manual)",
               "removed": "TODO", "remaining": "TODO"})

# Tag every extraction keyword each record matches (a record can match several)
b = blob_of(s).str.lower()
for theme, terms in EXTRACTION_KEYWORDS.items():
    s[f"kw_{theme}"] = b.str.contains(
        "|".join(re.escape(t.lower()) for t in terms), regex=True).astype(int)
s["Keywords_matched"] = s[[f"kw_{t}" for t in EXTRACTION_KEYWORDS]].apply(
    lambda r: "; ".join(t for t in EXTRACTION_KEYWORDS if r[f"kw_{t}"]), axis=1)

screening = pd.DataFrame(ledger)
screening.to_csv(INTERIM / "04_screening_ledger.csv", index=False)
print("Screening funnel")
display(screening)
print(f"\nRecords per extraction keyword (a record may match several).")
print(f"Paper: {PAPER_FUNNEL['retrieved']:,} retrieved -> {PAPER_FUNNEL['included']} included.\n")
display(s[[f"kw_{t}" for t in EXTRACTION_KEYWORDS]].sum()
        .rename(lambda x: x.replace("kw_", "")).rename("records").to_frame())

---
## A5 — Export and check the handover

A final validation that `corpus.csv` satisfies what notebooks 02–04 expect, so a
schema problem surfaces here rather than several notebooks later.

In [ ]:
cols = ["Title", "Abstract", "Keyword", "Year", "Journal", "Authors",
        "Affiliation", "DOI", "StudyType", "Keywords_matched", "SourceFile"]
corpus = s[cols].copy()
corpus["Year"] = corpus["Year"].astype(int)
corpus = corpus.reset_index(drop=True)

out = DATA_DIR / "corpus.csv"
corpus.to_csv(out, index=False)
print(f"Wrote {len(corpus):,} records to {out.resolve()}\n")

REQUIRED = ["Title", "Abstract", "Keyword", "Year", "Journal"]
c = pd.read_csv(out)
ok = True

missing = [x for x in REQUIRED if x not in c.columns]
print(f"[{'PASS' if not missing else 'FAIL'}] required columns present")
if missing:
    print(f"        missing: {missing}"); ok = False

yr = pd.to_numeric(c["Year"], errors="coerce")
bad = int((yr.isna() | ~yr.between(1900, 2100)).sum())
print(f"[{'PASS' if not bad else 'WARN'}] years parse and fall in range ({bad} problems)")

for col, label, thresh in [("Abstract", "abstracts", 0.5),
                           ("Journal", "journal names", 0.2),
                           ("Affiliation", "affiliations", 0.5)]:
    blank = c[col].isna().mean() if col in c.columns else 1.0
    tag = "PASS" if blank < thresh else "WARN"
    print(f"[{tag}] {label} populated ({blank:.0%} blank)")

print("\n" + ("Ready for notebooks 02, 03 and 04."
               if ok else "Fix the FAIL lines above before continuing."))
print(f"Records: {len(c):,}   Years: {int(yr.min())}-{int(yr.max())}")
display(corpus.head(3))

---

Next: **`02_bibliometric_analysis.ipynb`**.

**Search string** (Section 3.1.1), applied to PubMed, ScienceDirect and Consensus:

> `((((decision making) AND (urban planning)) AND (policy making)) AND (Artificial Intelligence))`

ScienceDirect was filtered to review and research articles, 2004–2024, in Social
Sciences, Environmental Science and Decision Sciences.

Citation: Lartey, D. & Law, K. M. Y. (2025). Artificial intelligence adoption in urban
planning governance: A systematic review of advancements in decision-making, and
policy making. *Landscape and Urban Planning*, 258, 105337.